# 导学版的视频没说这个脚本，不用看

In [1]:
############################## 示例数据集：西瓜数据
# 特征：[颜色(0:青绿,1:乌黑,2:浅白), 根蒂(0:蜷缩,1:稍蜷,2:硬挺), 敲声(0:浊响,1:沉闷,2:清脆)]
# 标签：1表示好瓜，0表示坏瓜
dataset = [
    [0, 0, 0, 1],
    [0, 0, 1, 1],
    [1, 0, 0, 1],
    [2, 1, 0, 0],
    [2, 2, 2, 0],
    [1, 2, 1, 0],
    [0, 1, 0, 1],
    [2, 1, 1, 0],
]
feature_names = ['color', 'root', 'knock']


##############################  计算基尼指数
from collections import Counter


def gini_index(groups):
    total_samples = sum(len(group) for group in groups)
    gini = 0.0
    for group in groups:
        size = len(group)
        if size == 0:
            continue
        score = 0.0
        labels = [row[-1] for row in group]
        label_counts = Counter(labels)
        for lbl in label_counts:
            p = label_counts[lbl] / size
            score += p * p
        gini += (1 - score) * (size / total_samples)
    return gini

##############################  寻找最佳划分
def split_dataset(index, value, dataset):
    left, right = [], []
    for row in dataset:
        if row[index] == value:
            left.append(row)
        else:
            right.append(row)
    return left, right

def get_best_split(dataset):
    best_index, best_value, best_score, best_groups = None, None, float('inf'), None
    n_features = len(dataset[0]) - 1
    for index in range(n_features):
        values = set(row[index] for row in dataset)
        for val in values:
            groups = split_dataset(index, val, dataset)
            gini = gini_index(groups)
            if gini < best_score:
                best_index, best_value, best_score, best_groups = index, val, gini, groups
    return {'index': best_index, 'value': best_value, 'groups': best_groups}

############################## 构建决策树（递归）
def to_terminal(group):
    labels = [row[-1] for row in group]
    return Counter(labels).most_common(1)[0][0]

def split(node, depth=0, max_depth=3, min_size=1):
    left, right = node['groups']
    del node['groups']
    
    # 终止条件
    if not left or not right:
        node['left'] = node['right'] = to_terminal(left + right)
        return
    if depth >= max_depth:
        node['left'], node['right'] = to_terminal(left), to_terminal(right)
        return
    if len(left) <= min_size:
        node['left'] = to_terminal(left)
    else:
        node['left'] = get_best_split(left)
        split(node['left'], depth + 1, max_depth, min_size)
    if len(right) <= min_size:
        node['right'] = to_terminal(right)
    else:
        node['right'] = get_best_split(right)
        split(node['right'], depth + 1, max_depth, min_size)

def build_tree(train, max_depth, min_size):
    root = get_best_split(train)
    split(root, 0, max_depth, min_size)
    return root

##############################  预测函数
def predict(node, row):
    if isinstance(node, dict):
        if row[node['index']] == node['value']:
            return predict(node['left'], row)
        else:
            return predict(node['right'], row)
    else:
        return node


##############################  构建树
tree = build_tree(dataset, max_depth=3, min_size=1)

# 测试预测
for row in dataset:
    y_true = row[-1]
    y_pred = predict(tree, row)
    print(f"True: {y_true}, Predicted: {y_pred}")

############################## 可选：可视化（简单打印树结构）
def print_tree(node, depth=0):
    if isinstance(node, dict):
        print(f"{'|  '*depth}[{feature_names[node['index']]} == {node['value']}]")
        print_tree(node['left'], depth + 1)
        print_tree(node['right'], depth + 1)
    else:
        print(f"{'|  '*depth}--> Predict: {node}")

print_tree(tree)

True: 1, Predicted: 1
True: 1, Predicted: 1
True: 1, Predicted: 1
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 1, Predicted: 1
True: 0, Predicted: 0
[color == 0]
|  [color == 0]
|  |  --> Predict: 1
|  |  --> Predict: 1
|  [root == 0]
|  |  --> Predict: 1
|  |  [color == 1]
|  |  |  --> Predict: 0
|  |  |  [color == 2]
|  |  |  |  --> Predict: 0
|  |  |  |  --> Predict: 0


In [5]:
from collections import Counter

labels = [1,1,0,1,0,0,0]

print(Counter(labels))

Counter({0: 4, 1: 3})


In [6]:
label_counts =  Counter(labels)
label_counts[1]

3

In [4]:
labels = [row[-1] for row in group]


NameError: name 'group' is not defined

In [2]:
label_counts

NameError: name 'label_counts' is not defined